# Final Submission Requirements: **Full RAG Pipeline**

✔ Contain a fully optimized RAG pipeline

✔ Work with multiple PDF documents

✔ Be tested on different prompts & retrieval tasks

✔ Use at least one open-source LLM (Mistral, Phi-2, TinyLlama, etc.)

✔ Implement at least one performance optimization (e.g., better chunking, hybrid retrieval, query expansion)

# Step 1: Install libraries and import as needed

In [ ]:
# Install required libraries with CUDA support
!pip install -q torch

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Check CUDA version first
!nvcc --version

# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

# install remaining libraries
!pip install llama-index
!pip install pymupdf
!pip install llama-index-llms-llama-cpp
!pip install llama-index-embeddings-huggingface

# Step 2: Download all open source LLMs

Mistral 7B, Phi-2, TinyLlama

In [ ]:
from llama_cpp import Llama
import os

In [ ]:
# Download Mistral model if not already present
model_path = "/content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

In [ ]:
# Download Phi-2 model if not already present
model_path = "/content/phi-2.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/phi-2-GGUF/resolve/main/phi-2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

In [ ]:
# Download TinyLlama model if not already present
model_path = "/content/tinyllama.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Step 3: Extract text using PyMuPDF

In [ ]:
import fitz  # PyMuPDF

# Define document paths
doc_paths = {
    "Unknown 1": "/content/sample_bank_statement.pdf",
    "Unknown 2": "/content/payslip_sample_image.pdf",
    "Unknown 3": "/content/appraisal_report.pdf",
    "Unknown 4": "/content/sample_contract.pdf",
    "Unknown 5": "/content/LenderFeesWorksheetNew.pdf"
}

# Extract text from all PDFs
doc_texts = {}

for i, (doc_type, path) in enumerate(doc_paths.items()):
    doc = fitz.open(path)
    text = "\n".join([page.get_text() for page in doc])
    doc_texts[f"Doc-{i+1}"] = text  # Temporarily label them "Unknown"
    print(f"Extracted {len(text.split())} words from {path}.")

# Step 4: Load the models with parameters

In [ ]:
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core import Document

# Load Mistral model with optimized generic parameters
mistral_llm = LlamaCPP(
    model_path="/content/mistral.gguf",
    temperature=0.0,  # Zero temperature for deterministic classification
    max_new_tokens=30,  # We only need a single category name
    context_window=4096,  # Increased context to handle our sampling approach
    verbose=True
)

# Load Phi-2 model with optimized generic parameters
phi2_llm = LlamaCPP(
    model_path="/content/phi-2.gguf",  # Path to your downloaded Phi-2 model
    temperature=0.0,                   # Deterministic output
    max_new_tokens=30,                # Short output (e.g. label, category, etc.)
    context_window=2048,             # Smaller than Mistral due to model size
    verbose=True

)

# Load TinyLlama model with suitable parameters
tinyllama_llm = LlamaCPP(
    model_path="/content/tinyllama.gguf",  # Path to TinyLlama GGUF file
    temperature=0.0,                       # Deterministic output
    max_new_tokens=30,                     # Keep output short
    context_window=2048,                   # Safe default for TinyLlama
    verbose=True
)

# Step 5: Define functions for classification of documents


In [ ]:
def prepare_document_for_classification(text):
    # Instead of truncating to first 500 chars, create a better representation

    # Get first, middle, and last portions
    doc_length = len(text)
    first_part = text[:min(500, doc_length)]

    middle_start = max(0, doc_length//2 - 250)
    middle_part = text[middle_start:middle_start + min(500, doc_length - middle_start)]

    last_start = max(0, doc_length - 500)
    last_part = text[last_start:]

    # Extract any structural elements (headings, tables, etc.)
    # This is a simplified version - could use regex for better extraction
    # possible_headers = [line.strip() for line in text.split('\n')
    #                   if line.strip() and len(line.strip()) < 50
    #                   and line.strip().isupper()]
    # headers = possible_headers[:10]  # Take first 10 potential headers

    return {
        "first_part": first_part,
        "middle_part": middle_part,
        "last_part": last_part,
        "total_length": doc_length,
        #"potential_headers": "\n".join(headers)
    }

**Change LLM value to go between different models as and when required**

In [ ]:
# Uncomment as needed

#llm = mistral_llm
llm = phi2_llm
#llm = tinyllama_llm

In [ ]:
def classify_document(text):
    doc_info = prepare_document_for_classification(text)

    prompt = f"""You are a document classification expert. Classify this document into one of these categories:
    - Bank Statement
    - Pay Slip
    - Appraisal Report
    - Service Agreement
    - Lender Fee Worksheet
    - Unknown

    Here's information extracted from the document:

    DOCUMENT START EXCERPT:
    {doc_info['first_part']}
    DOCUMENT START EXCERPT END

    DOCUMENT MIDDLE EXCERPT:
    {doc_info['middle_part']}
    DOCUMENT MIDDLE EXCERPT END

    DOCUMENT END EXCERPT:
    {doc_info['last_part']}
    DOCUMENT END EXCERPT END

    Total document length: {doc_info['total_length']} characters

    IMPORTANT INSTRUCTION: Your response must be EXACTLY ONE of these six options:
    Bank Statement
    Pay Slip
    Appraisal Report
    Service Agreement
    Lender Fee Worksheet
    Unknown

    Do not include any explanation, reasoning, or additional text. Respond with ONLY the category name.
    """

    response = llm.complete(prompt)
    raw_response = response.text.strip()

    # Post-process to extract just the category name
    categories = ["Bank Statement", "Pay Slip", "Appraisal Report", "Service Agreement", "Lender Fee Worksheet", "Unknown"]

    # First check if the response exactly matches one of our categories
    if raw_response in categories:
        return raw_response

    # If not, look for the category within the response
    for category in categories:
        if category.lower() in raw_response.lower():
            return category

    # If still no match, return the closest match
    import re
    words = re.findall(r'\b\w+\b', raw_response.lower())
    if "bank" in words or "statement" in words:
        return "Bank Statement"
    elif "pay" in words or "slip" in words or "salary" in words:
        return "Pay Slip"
    elif "appraisal" in words or "property" in words:
        return "Appraisal Report"
    elif "service" in words or "agreement" in words or "contract" in words:
        return "Service Agreement"
    elif "lender" in words or "fee" in words or "worksheet" in words or "borrower" in words:
        return "Lender Fee Worksheet"
    else:
        return "Unknown"

In [ ]:
# Classify each document
classified_docs = {}
for doc_id, text in doc_texts.items():
    doc_type = classify_document(text)
    classified_docs[doc_id] = {"text": text, "doc_type": doc_type}
    print(f"{doc_id} classified as: {doc_type}")


# Step 6: Perform chunking and word embedding of documents


In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Load embedding model
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# Create a custom sentence splitter
sentence_splitter = SentenceSplitter(
    chunk_size=512,  # The size of each chunk
    chunk_overlap=50  # The overlap between chunks
)

# Create separate indexes for each document type
index_map = {}

for doc_id, data in classified_docs.items():
    doc_type = data["doc_type"]

    if doc_type == "Unknown":
        continue  # Skip unknown documents

    document = Document(text=data["text"], metadata={"doc_type": doc_type})

    if doc_type not in index_map:
        index_map[doc_type] = VectorStoreIndex.from_documents(
            [document],
            embed_model=embed_model,
            transformations=[sentence_splitter]
        )
    else:
        nodes = sentence_splitter.get_nodes_from_documents([document])
        index_map[doc_type].insert_nodes(nodes)

    print(f"Indexed {doc_id} as {doc_type}.")

In [ ]:
# Create separate indexes for each document type
index_map = {}

for doc_id, data in classified_docs.items():
    doc_type = data["doc_type"]

    if doc_type == "Unknown":
        continue  # Skip unknown documents

    document = Document(text=data["text"], metadata={"doc_type": doc_type})

    if doc_type not in index_map:
        index_map[doc_type] = VectorStoreIndex.from_documents([document], embed_model=embed_model)
    else:
        index_map[doc_type].insert(document)

    print(f"Indexed {doc_id} as {doc_type}.")

# Step 7: Route the query to appropriate document and retrieve from relevant chunk


In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.response_synthesizers import CompactAndRefine
import re

In [ ]:
def route_query(query):
    # Check which document type the query is related to
    prompt = f"""
    Classify the following question into one of these categories:
    - 'Bank Statement'
    - 'Pay Slip'
    - 'Appraisal Report'
    - 'Service Agreement'
    - 'Lender Fee Worksheet'

    If it does not match any, respond with 'Unknown'.

    IMPORTANT INSTRUCTION: Your response must be EXACTLY ONE of these four options:
    Bank Statement
    Pay Slip
    Appraisal Report
    Service Agreement
    Lender Fee Worksheet
    Unknown

    Do not include any explanation, reasoning, or additional text. Respond with ONLY the category name.

    Query: {query}
    """

    doc_type = llm.complete(prompt).text.strip()

    raw_response = doc_type

    # Post-process to extract just the category name
    categories = ["Bank Statement", "Pay Slip", "Appraisal Report", "Service Agreement", "Lender Fee Worksheet", "Unknown"]

    # First check if the response exactly matches one of our categories
    if raw_response in categories:
        doc_type = raw_response

    # If not, look for the category within the response
    for category in categories:
        if category.lower() in raw_response.lower():
            doc_type = category

    # If still no match, return the closest match
    words = re.findall(r'\b\w+\b', raw_response.lower())
    if "bank" in words or "statement" in words:
        doc_type = "Bank Statement"
    elif "pay" in words or "slip" in words or "salary" in words:
        doc_type = "Pay Slip"
    elif "appraisal" in words or "property" in words:
        doc_type = "Appraisal Report"
    elif "service" in words or "agreement" in words or "contract" in words:
        return "Service Agreement"
    elif "lender" in words or "fee" in words or "worksheet" in words or "borrower" in words:
        return "Lender Fee Worksheet"
    else:
        doc_type = "Unknown"

    if doc_type not in index_map:
        return "Could not determine document type."

    # Get the correct index
    index = index_map[doc_type]

    # Create base retriever and query expansion retriever
    base_retriever = index.as_retriever(similarity_top_k=2)

    fusion_retriever = QueryFusionRetriever(
        retrievers=[base_retriever],
        llm=llm,
        similarity_top_k=2,
        num_queries=3,
        mode="reciprocal_rerank"
    )

    # Create synthesizer and engine
    response_synthesizer = CompactAndRefine(
        llm=llm
        )
    query_engine = RetrieverQueryEngine(
        retriever=fusion_retriever,
        response_synthesizer=response_synthesizer,
    )

    # Query and return
    response = query_engine.query(query)
    return f"📄 **Document Type:** {doc_type}\n🔍 **Answer:** {response}"

# Step 8: Test using different queries


In [ ]:
# Test different queries for Mistral 7B
print(route_query("What is my net salary?"))
print(route_query("What is the appraised value of the house?"))
print(route_query("What was my last deposit?"))

print(route_query("What is the total net salary for this month?"))
print(route_query("How much was the last transaction?"))
print(route_query("What is the estimated home value?"))

In [ ]:
# Test different queries for Phi-2
print(route_query("What is the refund policy in the contract?"))
print(route_query("How much is the total monthly loan payment?"))
print(route_query("What is the total net salary for this month?"))
print(route_query("How much was the last transaction?"))
print(route_query("What is the estimated home value?"))